<h1 style="
    background-color: #251351; 
    color: white; 
    padding: 10px; 
    text-align: left; 
    font-family: Arial, sans-serif; 
    font-size: 32px; 
    font-weight: bold; 
    margin: 10px 0;">
    Gerando DataFrames das Supercélulas
</h1>



In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

# ASE
from ase.io import read
from ase import Atoms, units

In [3]:
print(f'1 Ry em elétron-volt: {units.Ry} eV')
print(f'1 Raio de Bohr em Angstrom: {units.Bohr} Å')

1 Ry em elétron-volt: 13.605693012183622 eV
1 Raio de Bohr em Angstrom: 0.5291772105638411 Å


In [ ]:
def get_sc_data(path_dir: str | Path) -> pd.DataFrame:
    """Monta o DataFrame com as informações dos arquivos .out em um diretório de supercélulas."""
    path_dir = Path(path_dir)
    
    if not Path(path_dir).is_dir():
        raise NotADirectoryError(f"{path_dir.name} não é um diretório.")
    
    all_files = sorted(Path(path_dir).glob('*.out'))
    
    if len(all_files) == 0:
        raise FileNotFoundError(f"Nenhum arquivo .out encontrado em {path_dir}.")
    
    data = []
    for file in all_files:
        try:
            atoms = read(file)
            energy = atoms.get_total_energy()  # Energia total em eV
            volume = atoms.get_volume()        # Volume em Å³
            
            # lattice parameters
            lat_params = atoms.cell.cellpar()
            param_a = lat_params[0]
            param_c = lat_params[2]

        
            data.append({
                'Filename': file.name,
                'num_atomos': len(atoms),
                'E (eV)': energy,
                'V (Ang^3)': volume,
                'a (Ang)': param_a,
                'c (Ang)': param_c,
                
            })
        except Exception as e:
            print(f"Erro ao processar {file}: {e}")
    
    df = pd.DataFrame(data)
    return df

In [54]:
df_cell123 = get_sc_data(r'/home/jvc/QEspresso7.2/ZnO_database/scripts/anisotropic_strain-Aug22/cell123.out/')    

In [55]:
df_cell123

,Filename,E (eV),V (Ang^3),a (Ang),c (Ang),num_atomos
0,ZnO-2.94-1.45-123.out,-29485.916538,192.388560,2.942738,12.826712,24
1,ZnO-2.94-1.49-123.out,-29491.324665,196.663839,2.942738,13.111749,24
2,ZnO-2.94-1.52-123.out,-29495.843055,200.939162,2.942738,13.396788,24
3,ZnO-2.94-1.55-123.out,-29499.602299,205.214441,2.942738,13.681825,24
4,ZnO-2.94-1.58-123.out,-29502.716673,209.489764,2.942738,13.966865,24
...,...,...,...,...,...,...
116,ZnO-3.60-1.65-123.out,-29513.686224,398.095490,3.596679,17.767369,24
117,ZnO-3.60-1.68-123.out,-29512.683012,405.901243,3.596679,18.115747,24
118,ZnO-3.60-1.71-123.out,-29511.648166,413.707076,3.596679,18.464129,24
119,ZnO-3.60-1.74-123.out,-29510.592956,421.512829,3.596679,18.812507,24


In [45]:

df_cell123.loc[df_cell123['E (eV)'].idxmin(), 'E (eV)'] 



-29521.92861959363

In [ ]:
def get_supercell_info(path_diretorio: str | Path) -> pd.DataFrame:
    """Monta o DataFrame com as informações dos arquivos .out em um diretório de supercélulas."""
    
    if not Path(path_diretorio).is_dir():
        raise NotADirectoryError(f"{path_diretorio} não é um diretório válido.")
    
    diretorio = Path(path_diretorio)
    
    print(f"Lendo informações do diretório {diretorio.name}:\n{diretorio}\n")
    
    arquivos_de_saida: list[Path] = sorted(
        [file for file in diretorio.iterdir() if file.is_file() and file.suffix == '.out']
    )
    nx_ny_nz: str = diretorio.name.removeprefix('cell').removesuffix('.out')
    tamanho_supercelula = "x".join(nx_ny_nz)    

    dados = []
    mapa_tipo_atomo = {'Zn': 1, 'O': 2}

    for indice_arquivo, arquivo in enumerate(arquivos_de_saida):
        id_simulacao: str = arquivo.stem
        # print(f'\n{indice_arquivo+1}: {id_simulacao:_^{50}}\n')
        # TODO: Terminar aqui ? Retornando uma list[Atoms] ?
        
        atomos =  read(arquivo, format='espresso-out')
        
        # Informações em nível de sistema
        simbolos = atomos.get_chemical_symbols() 
        
        energia_total_eV = atomos.get_total_energy() 
        energia_total_Ry = energia_total_eV / units.Ry
        
        forcas_eV_Ang: np.ndarray = atomos.get_forces() 
        forcas_Ry_Bohr: np.ndarray = forcas_eV_Ang * (units.Bohr / units.Ry)
        
        # Params estruturais
        params = atomos.cell.cellpar()
        param_a = params[0]
        param_c = params[2]
        razao_ca = param_c / param_a

        volume_Ang = atomos.get_volume()
        volume_au = atomos.get_volume() / units.Bohr**3
        if table_mode == 'wide':
            # ===== simulacao a simulacao =========
            
            infos_simulacao = {
                'sim_id': id_simulacao,
                "supercelula": tamanho_supercelula,
                "num_atomos": len(atomos),
                'a': param_a,
                'c': param_c,
                'c/a': razao_ca,
                'energia_total_eV': energia_total_eV,
                'Volume_A³': volume_Ang,
                'energia_total_Ry': energia_total_Ry,
                'Volume_(a.u)^3': volume_au,
                'forcas_eV_Ang' : forcas_eV_Ang,
                'forcas_Ry_Bohr': forcas_Ry_Bohr
            }
            dados.append(infos_simulacao)
        else:    
            # =====  átomo a átomo =========
            numero_de_atomos = len(atomos) 
            for atom_index in range(numero_de_atomos):
                simbolo_atual = simbolos[atom_index]
                # Forças
                forca_atom_ev_ang = forcas_eV_Ang[atom_index]
                forca_atom_ry_bohr = forcas_Ry_Bohr[atom_index]
                
                modulo_forca_ev_ang = np.linalg.norm(forca_atom_ev_ang)
                modulo_forca_ry_bohr = np.linalg.norm(forca_atom_ry_bohr)

                # Monta o dicionário para a linha do DataFrame deste átomo
                linha_atomo = {
                    'sim_id': id_simulacao,
                    "supercelula": tamanho_supercelula,
                    "num_atomos": len(atomos),
                    'atom_id': f'atom {atom_index + 1}',  # ID do átomo (começando de 1)
                    'atom_type': mapa_tipo_atomo.get(simbolo_atual, -1), # Pega o tipo do mapa, -1 se não encontrar
                    'energia_total_eV': energia_total_eV,
                    'Volume_A³': volume_Ang,
                    'energia_total_Ry': energia_total_Ry,
                    'Volume_(a.u)^3': volume_au,
                    'fx_Ry_bohr': forca_atom_ry_bohr[0],
                    'fy_Ry_bohr': forca_atom_ry_bohr[1],
                    'fz_Ry_bohr': forca_atom_ry_bohr[2],
                    'modulo_Ry_bohr': modulo_forca_ry_bohr,
                    'modulo_eV_A': modulo_forca_ev_ang, 
                    'label_atom': simbolo_atual,
                }
                # Adiciona a linha (dicionário) à lista principal
                dados.append(linha_atomo) 
    return pd.DataFrame(dados)

In [12]:
# Wide dataset.
path = Path('../scripts/anisotropic_strain-Aug22')

cell_outputs: list[Path] = sorted(
    [Path(p) for p in path.iterdir() if p.name.endswith('.out')]
)

dataframes = []

for unit_cell in cell_outputs:
    cell_df = get_supercell_info(unit_cell, 'wide')
    dataframes.append(cell_df)

# Concatena todos em um único dataframe
base_de_dados_wide = pd.concat([df for df in dataframes], ignore_index=True)
base_de_dados_wide.to_csv('../data/dataset_wide.csv', index=False)

In [7]:
base_de_dados_wide

,sim_id,supercelula,a,c,c/a,energia_total_eV,Volume_A³,energia_total_Ry,Volume_(a.u)^3,forcas_eV_Ang,forcas_Ry_Bohr
0,ZnO-2.94-1.45-111,1x1x1,2.942738,4.275572,1.452923,-4914.318818,32.064749,-361.195774,216.383651,"[[-0.0, -0.0, -5.559851692105379], [0.0, 0.0, ...","[[-0.0, -0.0, -0.21624380374761829], [0.0, 0.0..."
1,ZnO-2.94-1.49-111,1x1x1,2.942738,4.370584,1.485210,-4915.220202,32.777295,-361.262024,221.192150,"[[-0.0, -0.0, -3.9539059476735328], [0.0, 0.0,...","[[-0.0, -0.0, -0.15378245844206787], [0.0, 0.0..."
2,ZnO-2.94-1.52-111,1x1x1,2.942738,4.465596,1.517497,-4915.973006,33.489841,-361.317355,226.000649,"[[0.0, 0.0, -2.6287269824469504], [0.0, -0.0, ...","[[0.0, 0.0, -0.10224120231578887], [0.0, -0.0,..."
3,ZnO-2.94-1.55-111,1x1x1,2.942738,4.560608,1.549784,-4916.599402,34.202387,-361.363394,230.809149,"[[0.0, 0.0, -1.543986529120231], [0.0, 0.0, -1...","[[0.0, 0.0, -0.06005151548666766], [0.0, 0.0, ..."
4,ZnO-2.94-1.58-111,1x1x1,2.942738,4.655621,1.582071,-4917.118557,34.914933,-361.401551,235.617648,"[[-0.0, -0.0, -0.6626717579144522], [0.0, -0.0...","[[-0.0, -0.0, -0.025773828062898983], [0.0, -0..."
...,...,...,...,...,...,...,...,...,...,...,...
2294,ZnO-3.60-1.65-331,3x3x1,10.790039,5.922458,0.548882,-44270.535791,597.143099,-3253.824392,4029.721381,"[[0.00035198402278003893, 0.000203374260057714...","[[1.3689998971091494e-05, 7.90999940550283e-06..."
2295,ZnO-3.60-1.68-331,3x3x1,10.790039,6.038580,0.559644,-44269.031624,608.851361,-3253.713838,4108.732647,"[[6.890556472246198e-05, 3.9852098999931365e-0...","[[2.6799997985774445e-06, 1.5499998835056112e-..."
2296,ZnO-3.60-1.71-331,3x3x1,10.790039,6.154714,0.570407,-44267.479969,620.560710,-3253.599793,4187.751254,"[[-6.170647587086147e-05, -3.573833394187393e-...","[[-2.3999998196215915e-06, -1.3899998955308385..."
2297,ZnO-3.60-1.74-331,3x3x1,10.790039,6.270836,0.581169,-44265.896994,632.268972,-3253.483447,4266.762520,"[[7.404777104503376e-05, 4.268031247734585e-05...","[[2.87999978354591e-06, 1.6599998752382675e-06..."


In [13]:
# Long dataset (análise forças).
dataframes = []

for unit_cell in cell_outputs:
    cell_df = get_supercell_info(unit_cell)
    dataframes.append(cell_df)

# Concatena todos em um único dataframe
base_de_dados_long = pd.concat([df for df in dataframes], ignore_index=True)
base_de_dados_long.to_csv('../data/dataset_long.csv', index=False)

In [15]:
base_de_dados_long.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43560 entries, 0 to 43559
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   sim_id            43560 non-null  object 
 1   supercelula       43560 non-null  object 
 2   atom_id           43560 non-null  object 
 3   atom_type         43560 non-null  int64  
 4   energia_total_eV  43560 non-null  float64
 5   Volume_A³         43560 non-null  float64
 6   energia_total_Ry  43560 non-null  float64
 7   Volume_(a.u)^3    43560 non-null  float64
 8   fx_Ry_bohr        43560 non-null  float64
 9   fy_Ry_bohr        43560 non-null  float64
 10  fz_Ry_bohr        43560 non-null  float64
 11  modulo_Ry_bohr    43560 non-null  float64
 12  modulo_eV_A       43560 non-null  float64
 13  label_atom        43560 non-null  object 
dtypes: float64(9), int64(1), object(4)
memory usage: 4.7+ MB
